In [44]:
from trino.dbapi import connect
import pandas as pd

# Conectando no TRINO (Porta 8080)
conn = connect(
    host='localhost',
    port=8080,
    user='postgres', # Trino não exige senha por padrão em dev
    catalog='pg', # O nome do arquivo .properties que criamos
    schema='public'
)

# O Trino traduz esse SQL e manda pro Postgres
query = "SELECT * FROM funcionarios LIMIT 10"

df_trino = pd.read_sql(query, conn)
df_trino

/var/folders/hr/hf99z3sn635_fs6qx5t20m340000gn/T/ipykernel_37061/2347547051.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trino = pd.read_sql(query, conn)


,id,nome,cargo
0,1,Ana,Ana
1,2,João,PM
2,3,Bianca,CEO
3,5,Andrei Também,Fundador


# Exercicio

## Subir todas as tabelas do escola_v2.db com os dados para o Postgres

## Trino join 

In [38]:
select_query = """
SELECT 
    clientes_crm.gerente_conta
FROM 
    pg.public.clientes_crm AS clientes_crm       -- Fonte 1: Postgres
"""

In [39]:
df_join = pd.read_sql(select_query, conn)
df_join

/var/folders/hr/hf99z3sn635_fs6qx5t20m340000gn/T/ipykernel_37061/901126344.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_join = pd.read_sql(select_query, conn)


,gerente_conta
0,Ana
1,Ana
2,Bruno
3,Carla
4,Bruno


In [42]:
# Conexão padrão para consulta (usando trino.dbapi ou sqlalchemy)
conn = connect(
    host='localhost',
    port=8080,
    user='admin',
    catalog='crm',
    schema='public'
)

query = """
SELECT 
    c.segmento,
    c.gerente_conta,
    v.total_gasto,
    v.data_ultima_compra
FROM 
    pg.public.clientes_crm AS c      -- Fonte 1: Postgres (Persistente)
JOIN 
    memory.default.vendas_csv AS v          -- Fonte 2: CSV na RAM (Volátil)
    ON c.custkey = v.custkey
WHERE 
    v.total_gasto > 100
ORDER BY 
    v.total_gasto DESC
"""

df_resultado = pd.read_sql(query, conn)
display(df_resultado)

/var/folders/hr/hf99z3sn635_fs6qx5t20m340000gn/T/ipykernel_37061/3863911537.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_resultado = pd.read_sql(query, conn)


,segmento,gerente_conta,total_gasto,data_ultima_compra
0,VIP,Ana,1200.5,2023-02-15
1,VIP,Ana,500.0,2023-01-01


## Conceitos de Otimização (Aplicados a esse cenário)

Pushdown:

Se rodarmos WHERE v.valor > 1000 (filtro no MySQL), o Trino é inteligente. Ele pede pro MySQL: "Me manda só as vendas acima de 1000". Isso economiza rede.

Cross-Database Join:

Se a tabela do MySQL tivesse 1 bilhão de linhas e a do Postgres 10 linhas, o Trino tentaria jogar a tabela pequena (Postgres) para todos os nós e escanear a grande (MySQL).

### Dica: Sempre filtre o máximo possível usando WHERE antes de fazer o JOIN entre bancos diferentes.